# 8-Puzzle Game trong Jupyter Notebook

Chạy cell bên dưới để chơi trò 8-puzzle. Có giao diện nút bấm nếu môi trường có `ipywidgets`; nếu không, chương trình sẽ chuyển sang chế độ nhập lệnh bằng console.


In [5]:

# 8-Puzzle Game - chạy trong Jupyter Notebook
# Cách dùng:
# 1. Chạy cell này.
# 2. Bấm các nút Lên/Xuống/Trái/Phải để di chuyển ô trống.
# 3. Bấm "Giải bằng AI" để xem lời giải tự động.

import random
import heapq
from collections import deque

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False


GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)   # 0 là ô trống


def print_board(state):
    """In bàn cờ dạng text, dùng được cả khi không có ipywidgets."""
    for i in range(0, 9, 3):
        row = state[i:i+3]
        print(" ".join("_" if x == 0 else str(x) for x in row))
    print()


def inversion_count(state):
    """
    Đếm số nghịch thế.
    Với 8-puzzle 3x3: trạng thái giải được khi số nghịch thế là chẵn.
    """
    arr = [x for x in state if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv


def is_solvable(state):
    """Kiểm tra trạng thái có giải được không."""
    return inversion_count(state) % 2 == 0


def generate_solvable_state(shuffle_steps=100):
    """
    Sinh trạng thái ngẫu nhiên nhưng chắc chắn giải được.
    Cách làm: bắt đầu từ GOAL rồi di chuyển hợp lệ ngẫu nhiên nhiều lần.
    """
    state = GOAL
    last_move = None

    opposite = {
        "up": "down",
        "down": "up",
        "left": "right",
        "right": "left"
    }

    for _ in range(shuffle_steps):
        possible = valid_moves(state)
        if last_move is not None and opposite[last_move] in possible and len(possible) > 1:
            possible.remove(opposite[last_move])

        move = random.choice(possible)
        state = move_blank(state, move)
        last_move = move

    return state


def valid_moves(state):
    """
    Trả về danh sách hướng có thể di chuyển ô trống.
    Lưu ý: "up" nghĩa là ô trống đi lên.
    """
    blank = state.index(0)
    row, col = divmod(blank, 3)

    moves = []
    if row > 0:
        moves.append("up")
    if row < 2:
        moves.append("down")
    if col > 0:
        moves.append("left")
    if col < 2:
        moves.append("right")

    return moves


def move_blank(state, direction):
    """Di chuyển ô trống theo hướng direction."""
    if direction not in valid_moves(state):
        return state

    blank = state.index(0)
    row, col = divmod(blank, 3)

    if direction == "up":
        target = blank - 3
    elif direction == "down":
        target = blank + 3
    elif direction == "left":
        target = blank - 1
    elif direction == "right":
        target = blank + 1
    else:
        return state

    new_state = list(state)
    new_state[blank], new_state[target] = new_state[target], new_state[blank]
    return tuple(new_state)


def manhattan_distance(state):
    """
    Heuristic Manhattan dùng cho A*.
    Tổng khoảng cách từng ô số từ vị trí hiện tại tới vị trí đích.
    """
    distance = 0

    for index, value in enumerate(state):
        if value == 0:
            continue

        current_row, current_col = divmod(index, 3)
        goal_index = GOAL.index(value)
        goal_row, goal_col = divmod(goal_index, 3)

        distance += abs(current_row - goal_row) + abs(current_col - goal_col)

    return distance


def get_neighbors(state):
    """Sinh các trạng thái kề từ một trạng thái."""
    result = []
    for direction in valid_moves(state):
        next_state = move_blank(state, direction)
        result.append((next_state, direction))
    return result


def solve_astar(start):
    """
    Giải 8-puzzle bằng thuật toán A*.
    f(n) = g(n) + h(n)
    g(n): số bước đã đi
    h(n): Manhattan distance
    """
    if not is_solvable(start):
        return None

    priority_queue = []
    heapq.heappush(priority_queue, (manhattan_distance(start), 0, start, []))

    visited_cost = {start: 0}

    while priority_queue:
        f, g, state, path = heapq.heappop(priority_queue)

        if state == GOAL:
            return path

        for next_state, move in get_neighbors(state):
            new_g = g + 1

            if next_state not in visited_cost or new_g < visited_cost[next_state]:
                visited_cost[next_state] = new_g
                new_f = new_g + manhattan_distance(next_state)
                heapq.heappush(priority_queue, (new_f, new_g, next_state, path + [move]))

    return None


if HAS_WIDGETS:
    class EightPuzzleGame:
        def __init__(self):
            self.state = generate_solvable_state()
            self.steps = 0
            self.message = "Bắt đầu chơi! Hãy đưa bàn cờ về trạng thái 1 2 3 / 4 5 6 / 7 8 _"

            self.output = widgets.Output()

            self.btn_up = widgets.Button(description="⬆ Lên", button_style="info")
            self.btn_down = widgets.Button(description="⬇ Xuống", button_style="info")
            self.btn_left = widgets.Button(description="⬅ Trái", button_style="info")
            self.btn_right = widgets.Button(description="➡ Phải", button_style="info")
            self.btn_new = widgets.Button(description=" Ván mới", button_style="warning")
            self.btn_solve = widgets.Button(description=" Giải bằng AI", button_style="success")

            self.btn_up.on_click(lambda b: self.play("up"))
            self.btn_down.on_click(lambda b: self.play("down"))
            self.btn_left.on_click(lambda b: self.play("left"))
            self.btn_right.on_click(lambda b: self.play("right"))
            self.btn_new.on_click(lambda b: self.new_game())
            self.btn_solve.on_click(lambda b: self.show_solution())

            controls = widgets.VBox([
                widgets.HBox([self.btn_up]),
                widgets.HBox([self.btn_left, self.btn_down, self.btn_right]),
                widgets.HBox([self.btn_new, self.btn_solve])
            ])

            display(controls)
            display(self.output)
            self.render()

        def board_html(self):
            cells = ""
            for value in self.state:
                if value == 0:
                    cells += """
                    <div style="
                        width:80px;height:80px;
                        border:2px dashed #999;
                        display:flex;align-items:center;justify-content:center;
                        font-size:28px;border-radius:12px;background:#f7f7f7;color:#777777;">
                        _
                    </div>
                    """
                else:
                    cells += f"""
                    <div style="
                        width:80px;height:80px;
                        border:2px solid #333;
                        display:flex;align-items:center;justify-content:center;
                        font-size:32px;font-weight:bold;
                        border-radius:12px;background:#ffffff;color:#111111;">
                        {value}
                    </div>
                    """

            return f"""
            <div style="font-family:Arial, sans-serif;">
                <h2>Trò chơi 8-Puzzle</h2>
                <p><b>Số bước:</b> {self.steps}</p>
                <p><b>Thông báo:</b> {self.message}</p>
                <div style="
                    display:grid;
                    grid-template-columns:repeat(3, 80px);
                    gap:8px;
                    margin:12px 0;">
                    {cells}
                </div>
                <p><b>Trạng thái đích:</b></p>
                <pre style="font-size:18px;">1 2 3
4 5 6
7 8 _</pre>
            </div>
            """

        def render(self):
            with self.output:
                clear_output(wait=True)
                display(HTML(self.board_html()))

        def play(self, direction):
            if self.state == GOAL:
                self.message = "Bạn đã thắng rồi! Bấm Ván mới để chơi tiếp."
                self.render()
                return

            old_state = self.state
            self.state = move_blank(self.state, direction)

            if self.state != old_state:
                self.steps += 1
                self.message = f"Bạn vừa đi: {direction}"

                if self.state == GOAL:
                    self.message = f"🎉 Chúc mừng! Bạn đã giải xong trong {self.steps} bước."
            else:
                self.message = f"Không thể đi hướng: {direction}"

            self.render()

        def new_game(self):
            self.state = generate_solvable_state()
            self.steps = 0
            self.message = "Đã tạo ván mới!"
            self.render()

        def show_solution(self):
            path = solve_astar(self.state)

            if path is None:
                self.message = "Trạng thái này không giải được."
            elif len(path) == 0:
                self.message = "Bàn cờ đang ở trạng thái đích rồi."
            else:
                vietnamese = {
                    "up": "Lên",
                    "down": "Xuống",
                    "left": "Trái",
                    "right": "Phải"
                }
                path_text = " → ".join(vietnamese[x] for x in path)
                self.message = f"AI tìm được lời giải {len(path)} bước: {path_text}"

            self.render()


    game = EightPuzzleGame()

else:

    print("Môi trường hiện tại chưa có ipywidgets.")
    print("Bạn vẫn có thể chơi bằng cách nhập: up, down, left, right, solve, new, quit")
    print()

    state = generate_solvable_state()
    steps = 0

    while True:
        print_board(state)
        print(f"Số bước: {steps}")

        if state == GOAL:
            print("🎉 Bạn đã thắng!")
            break

        command = input("Nhập lệnh up/down/left/right/solve/new/quit: ").strip().lower()

        if command == "quit":
            print("Đã thoát trò chơi.")
            break
        elif command == "new":
            state = generate_solvable_state()
            steps = 0
        elif command == "solve":
            path = solve_astar(state)
            if path is None:
                print("Trạng thái này không giải được.")
            else:
                print("Lời giải AI:", " -> ".join(path))
                print("Số bước tối ưu:", len(path))
        elif command in ["up", "down", "left", "right"]:
            new_state = move_blank(state, command)
            if new_state == state:
                print("Không đi được hướng đó.")
            else:
                state = new_state
                steps += 1
        else:
            print("Lệnh không hợp lệ.")


Output()